# VI. Regression Analysis — dự đoán giá quán ăn Hòa Lạc

**Mục tiêu:** dự đoán chi phí cho **hai người**, đơn vị **VND**, bằng đúng hai mô hình:
**Multiple Linear Regression** và **Random Forest Regressor**. Không dự đoán điểm đánh giá.

**Dữ liệu:** SQL Server `localhost` → database `RestaurantAnalysis` → ba bảng
`dbo.Restaurant`, `dbo.Location`, `dbo.Review` theo cấu trúc report. Nối qua
`location_id` và `restaurant_id` để lấy 17 thuộc tính phục vụ phân tích. Có 1.210 quán: 1.000 nội thành và 210 thuộc
cụm Hòa Lạc–Thạch Thất. Nhóm 210 gồm 10 địa phương, không chỉ riêng 90 dòng mang tên Hòa Lạc.

**Cách chạy:** chọn kernel Python 3, chạy các ô từ trên xuống. Ô đầu cài thư viện Python;
máy cần có SQL Server đang chạy và ODBC Driver 18 for SQL Server. SSMS là công cụ quản lý,
notebook kết nối vào SQL Server chứ không kết nối vào cửa sổ SSMS.

Chạy file `RestaurantAnalysis_Hanoi_Restaurants.sql` trong SSMS trước để tạo đủ ba bảng.
Notebook chỉ đọc các bảng có sẵn; không nhập lại, xóa hay sửa dữ liệu SQL. Các ô cuối lưu
kết quả vào thư mục `ket_qua_regression` cạnh vị trí chạy notebook. Không có biểu đồ.

Mỗi ô code bên dưới có **một nhiệm vụ chính**, kèm giải thích đầu vào, đầu ra và cách tính.

## Ô 1 — Cài thư viện
**Công dụng:** cài các thư viện cần thiết vào đúng kernel đang dùng.
Nếu thư viện đã có thì pip giữ phiên bản đáp ứng yêu cầu. Đây không phải bước tải dữ liệu.

In [1]:
%pip install -q pandas numpy scikit-learn pyodbc

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\NEO 16\Documents\Codex\2026-09-21\singular-value-decomposition-l-g\work\regression-env\Scripts\python.exe -m pip install --upgrade pip


## Ô 2 — Nhập thư viện
**Công dụng:** nạp công cụ đọc SQL, xử lý dữ liệu, mã hóa đầu vào, huấn luyện và đánh giá.
`clone` tạo một mô hình mới chưa học; `Pipeline` giữ bước chuẩn bị dữ liệu đi cùng mô hình,
để từng lượt kiểm định chỉ học cách chuẩn hóa và từ điển món ăn từ dữ liệu train của lượt đó.

In [2]:
from pathlib import Path
from contextlib import closing
import numpy as np
import pandas as pd
import pyodbc
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Ô 3 — Kết nối SQL Server và đọc dữ liệu
**Công dụng:** dùng Windows Authentication để đọc ba bảng theo schema trong report.
Chỉ sửa `SERVER` và `DATABASE` nếu chuyển sang máy khác. Không cần ghi mật khẩu trong notebook.

- `Restaurant`: giá `average_cost_for_two`, loại món, dịch vụ, tọa độ và khóa `location_id`.
- `Location`: nối bằng `location_id` để lấy `locality`, `locality_verbose`.
- `Review`: nối bằng `restaurant_id` để lấy điểm đánh giá và số lượt đánh giá.
- Cờ dịch vụ trong `Restaurant` là `Yes/No`; truy vấn chuyển thành `BIT` (0/1) cho mô hình.
- `location_id` chỉ là khóa nối bảng, không phải biến số đầu vào của mô hình.

Truy vấn trả 17 cột như dữ liệu phân tích ban đầu, theo thứ tự `restaurant_id` để giữ cách chia mẫu.
`closing` đóng kết nối sau khi lấy dữ liệu xong.
`TrustServerCertificate=yes` dùng cho chứng chỉ của SQL Server cục bộ trong bài thực hành này.


In [3]:
SERVER = "localhost"
DATABASE = "RestaurantAnalysis"
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={SERVER};DATABASE={DATABASE};"
    "Trusted_Connection=yes;Encrypt=yes;TrustServerCertificate=yes;"
)
query = """
SELECT r.restaurant_id, r.restaurant_name, r.address,
       l.locality, l.locality_verbose, r.cuisines,
       r.average_cost_for_two, r.currency,
       CAST(CASE WHEN r.has_table_booking = 'Yes' THEN 1 ELSE 0 END AS BIT) AS has_table_booking,
       CAST(CASE WHEN r.has_online_delivery = 'Yes' THEN 1 ELSE 0 END AS BIT) AS has_online_delivery,
       CAST(CASE WHEN r.is_delivering_now = 'Yes' THEN 1 ELSE 0 END AS BIT) AS is_delivering_now,
       r.price_range, t.aggregate_rating, t.rating_text, t.votes,
       r.latitude, r.longitude
FROM dbo.Restaurant AS r
JOIN dbo.Location AS l ON l.location_id = r.location_id
JOIN dbo.Review AS t ON t.restaurant_id = r.restaurant_id
ORDER BY r.restaurant_id;
"""
with closing(pyodbc.connect(connection_string, timeout=10)) as connection:
    cursor = connection.cursor()
    cursor.execute(query)
    df = pd.DataFrame.from_records(cursor.fetchall(), columns=[c[0] for c in cursor.description])

print(f"Đã đọc {len(df):,} quán, {df.shape[1]} cột từ {SERVER}/{DATABASE}.")
display(df.head(3))

Đã đọc 1,210 quán, 17 cột từ localhost/RestaurantAnalysis.


,restaurant_id,restaurant_name,address,locality,locality_verbose,cuisines,average_cost_for_two,currency,has_table_booking,has_online_delivery,is_delivering_now,price_range,aggregate_rating,rating_text,votes,latitude,longitude
0,HN0001,Steakhouse & Wine Bách Khoa - Cầu Giấy,"Số 92 Đường Cầu Giấy, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Đồ Âu, bít tết, rượu vang",576000,VND,True,True,True,4,4.5,Xuất sắc,727,21.027111,105.794479
1,HN0002,Bún Đậu Minh Khang - Duy Tân,"Số 191 Phố Duy Tân, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Món Việt, quán bún đậu",107000,VND,False,True,True,1,3.8,Tốt,468,21.026756,105.796738
2,HN0003,Pizza & Pasta Trung Hòa - Tô Hiệu,"Số 32 Phố Tô Hiệu, Quận Cầu Giấy, Hà Nội",Cầu Giấy,"Quận Cầu Giấy, Hà Nội","Món Ý, Đồ Âu",167000,VND,False,True,True,2,3.9,Tốt,599,21.034592,105.796135


## Ô 4 — Kiểm tra dữ liệu và xác định khu vực
**Công dụng:** bảo đảm dữ liệu hợp lệ trước khi học và gán đúng nhóm theo báo cáo.

- Bỏ dòng thiếu đầu vào, giá không dương, điểm ngoài (0; 5], votes âm hoặc tọa độ không hợp lệ.
  In số dòng bị loại để không bỏ dữ liệu âm thầm. Giữ giá cao hợp lệ, không tự xóa bằng IQR.
- Dừng nếu trùng mã, có tiền tệ khác VND hoặc địa phương ngoài danh sách đã định nghĩa.
- `region` chỉ dùng chia mẫu và lọc kết quả, không phải giá hay nhãn được học.
- Chuỗi món ăn được viết thường và tách bằng `|`; ví dụ `Món Việt, Lẩu` thành `món việt|lẩu`.
- Điểm đánh giá vốn có một chữ số thập phân; làm tròn lại một chữ số để tránh sai số lưu trữ
  SQL `float` khiến 4,2 thành 4,1999998 khi xét ngưỡng.

In [4]:
HOA_LAC = ["Hòa Lạc", "Thạch Hòa", "Tân Xã", "Bắc Phú Cát", "Thạch Thất",
           "Liên Quan", "Tây Phương", "Yên Bình", "Bình Yên", "Phú Cát"]
INNER_CITY = ["Cầu Giấy", "Đống Đa", "Hoàn Kiếm", "Hai Bà Trưng", "Ba Đình",
              "Thanh Xuân", "Tây Hồ", "Nam Từ Liêm", "Hà Đông", "Hoàng Mai",
              "Long Biên", "Bắc Từ Liêm"]
numeric_features = ["latitude", "longitude", "aggregate_rating", "votes",
                    "has_table_booking", "has_online_delivery"]
target = "average_cost_for_two"
data = df.copy()

if data["restaurant_id"].isna().any() or data["restaurant_id"].duplicated().any():
    raise ValueError("Restaurant ID bị thiếu hoặc trùng. Kiểm tra nguồn SQL trước khi huấn luyện.")
if not data["currency"].astype("string").str.strip().eq("VND").all():
    raise ValueError("Bài này chỉ nhận giá VND; cần xử lý tiền tệ khác trước khi học.")
data["locality"] = data["locality"].astype("string").str.strip()
if not data["locality"].isin(HOA_LAC + INNER_CITY).all():
    raise ValueError("Có khu vực chưa nằm trong danh sách 22 địa phương của báo cáo.")
for column in numeric_features + [target]:
    data[column] = pd.to_numeric(data[column], errors="coerce")
data["aggregate_rating"] = data["aggregate_rating"].round(1)
data[numeric_features + [target]] = data[numeric_features + [target]].replace([np.inf, -np.inf], np.nan)
data["cuisines"] = (data["cuisines"].astype("string").str.strip().str.casefold()
                    .str.replace(r"\s*,\s*", "|", regex=True))
data = data.dropna(subset=numeric_features + [target, "cuisines"])
valid = ((data[target] > 0) & (data["aggregate_rating"] > 0) & (data["aggregate_rating"] <= 5)
         & (data["votes"] >= 0) & data["latitude"].between(-90, 90)
         & data["longitude"].between(-180, 180) & data["cuisines"].ne("")
         & data["has_table_booking"].isin([0, 1]) & data["has_online_delivery"].isin([0, 1]))
data = data.loc[valid].copy().reset_index(drop=True)
data["region"] = np.where(data["locality"].isin(HOA_LAC), "Cụm Hòa Lạc", "Nội thành")
if len(data) < 10 or data["region"].nunique() != 2:
    raise ValueError("Không đủ dữ liệu hợp lệ của cả hai khu vực để chạy bài này.")

print(f"Dùng {len(data)} quán; loại {len(df) - len(data)} dòng không hợp lệ.")
display(data.groupby("region")[target].agg(so_quan="size", gia_trung_binh="mean", trung_vi="median").round(0))

Dùng 1210 quán; loại 0 dòng không hợp lệ.


,so_quan,gia_trung_binh,trung_vi
region,,,
Cụm Hòa Lạc,210,201262.0,112500.0
Nội thành,1000,269024.0,215000.0


## Ô 5 — Tạo X, y và chia train/test
**Công dụng:** tạo đầu vào, đầu ra và giữ riêng 20% dữ liệu để kiểm tra cuối cùng.

**X:** tọa độ, điểm đánh giá, votes, đặt bàn, giao hàng và loại món.
**y:** `average_cost_for_two` (VND/2 người).
Không đưa `price_range` vào X vì được suy ra từ chính giá; không đưa mã, tên quán hoặc
`rating_text` (đã có điểm đánh giá tương ứng). Mô hình dành cho quán đã có đánh giá, không phải quán mới chưa có rating/votes.

`stratify=region` giữ tỷ lệ hai khu vực. Với dữ liệu hiện có: train 968 = 800 nội thành + 168 Hòa Lạc;
test 242 = 200 nội thành + 42 Hòa Lạc. `random_state=42` giúp lặp lại đúng cách chia.
Đây là kiểm tra quán mới trong hai thị trường đã có dữ liệu, không phải thử chuyển mô hình từ nội thành sang toàn bộ Hòa Lạc chưa từng thấy.

In [5]:
features = numeric_features + ["cuisines"]
X = data[features].copy()
y = data[target].astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=data["region"]
)
train_regions = data.loc[X_train.index, "region"]
test_regions = data.loc[X_test.index, "region"]
print("Số dòng train/test:", len(X_train), "/", len(X_test))
display(pd.DataFrame({"Train": train_regions.value_counts(), "Test": test_regions.value_counts()}))

Số dòng train/test: 968 / 242


,Train,Test
region,,
Nội thành,800,200
Cụm Hòa Lạc,168,42


## Ô 6 — Định nghĩa cách mã hóa đầu vào
**Công dụng:** mô tả cách chuyển X thành ma trận số; chưa học từ test.

- Biến số: `z = (x − trung bình train) / độ lệch chuẩn train`.
- Loại món: mỗi món là một cột 0/1. `món việt|lẩu` bật hai cột tương ứng; món chưa có trong train bị bỏ qua.
- Từ điển món và tham số chuẩn hóa được học bên trong Pipeline, chỉ từ train của từng lượt.
- Chia train thành 5 phần giữ tỷ lệ khu vực: mỗi lượt học 4 phần, kiểm tra phần còn lại.
  Hai mô hình dùng đúng cùng 5 lượt chia; không dùng test để chọn mô hình.

In [6]:
preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("cuisines", CountVectorizer(token_pattern=r"[^|]+", lowercase=False, binary=True), "cuisines")
])
cv = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_train, train_regions))
print("Đã chuẩn bị cách mã hóa và 5 lượt kiểm định chéo trên train.")

Đã chuẩn bị cách mã hóa và 5 lượt kiểm định chéo trên train.


## Ô 7 — Huấn luyện Linear Regression
**Công dụng:** học mô hình hồi quy tuyến tính đa biến và đo MAE kiểm định chéo trên train.

$$\hat y = b_0 + b_1x_1 + \cdots + b_px_p$$

Mô hình chọn hệ số để tối thiểu hóa tổng bình phương sai số trên tập học.
Các $x$ ở đây là biến số đã chuẩn hóa và cột món ăn 0/1, không phải toàn bộ giá trị thô.
`fit` học từ train; `cross_val_score` tự tạo các bản sao và học lại trong từng lượt CV.
Scikit-learn trả `neg_mean_absolute_error`; đổi dấu để có MAE dương, rồi lấy trung bình 5 lượt.

In [7]:
lr_model = Pipeline([("prepare", clone(preprocessor)), ("model", LinearRegression())])
lr_cv_mae = -cross_val_score(lr_model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
lr_model.fit(X_train, y_train)
print(f"Linear Regression — MAE CV trung bình: {lr_cv_mae.mean():,.0f} VND")
print("MAE từng lượt:", np.round(lr_cv_mae, 0))

Linear Regression — MAE CV trung bình: 45,437 VND
MAE từng lượt: [47869. 45187. 44067. 45470. 44593.]


## Ô 8 — Huấn luyện Random Forest
**Công dụng:** học mô hình rừng ngẫu nhiên để so sánh với Linear Regression.

$$\hat y = \frac{1}{T}\sum_{t=1}^{T} f_t(x)$$

Mỗi cây chia dữ liệu thành các nhóm; giá dự đoán ở lá là trung bình giá các mẫu thuộc lá đó.
Rừng lấy trung bình dự đoán của **200 cây**. `min_samples_leaf=3` yêu cầu ít nhất 3 mẫu ở mỗi lá,
giúp hạn chế việc học thuộc từng quán. `random_state=42` cố định ngẫu nhiên;
`n_jobs=-1` cho phép dùng các CPU có sẵn. Cách chuẩn hóa không làm thay đổi bản chất các ngưỡng chia của cây.
Các tham số được đặt trước, chưa dò tìm tham số hoặc khẳng định đây là cấu hình tối ưu.

In [8]:
rf_model = Pipeline([
    ("prepare", clone(preprocessor)),
    ("model", RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42, n_jobs=-1))
])
rf_cv_mae = -cross_val_score(rf_model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
rf_model.fit(X_train, y_train)
print(f"Random Forest — MAE CV trung bình: {rf_cv_mae.mean():,.0f} VND")
print("MAE từng lượt:", np.round(rf_cv_mae, 0))

Random Forest — MAE CV trung bình: 43,940 VND
MAE từng lượt: [44476. 39674. 43286. 52911. 39350.]


## Ô 9 — Chọn mô hình bằng dữ liệu train
**Công dụng:** cố định mô hình sẽ dùng cho bảng giá tham chiếu trước khi xem kết quả test.
Chọn mô hình có MAE trung bình 5 lượt CV thấp hơn; MAE có đơn vị VND nên dễ gắn với sai lệch giá.
Test không tham gia quyết định này. Nếu hòa, giữ thứ tự Linear Regression trước.

In [9]:
models = {"Linear Regression": lr_model, "Random Forest": rf_model}
cv_comparison = pd.DataFrame({
    "MAE CV (VND)": [lr_cv_mae.mean(), rf_cv_mae.mean()],
    "Độ lệch chuẩn MAE CV (VND)": [lr_cv_mae.std(ddof=1), rf_cv_mae.std(ddof=1)]
}, index=list(models))
best_name = cv_comparison["MAE CV (VND)"].idxmin()
display(cv_comparison.round(0))
print("Mô hình được chọn từ CV trên train:", best_name)

,MAE CV (VND),Độ lệch chuẩn MAE CV (VND)
Linear Regression,45437.0,1463.0
Random Forest,43940.0,5488.0


Mô hình được chọn từ CV trên train: Random Forest


## Ô 10 — Đánh giá MAE, RMSE, R² và xem dự đoán trên test
**Công dụng:** so sánh hai mô hình bằng các giá chưa dùng để huấn luyện.
Với $n$ quán, giá thật $y_i$ và giá dự đoán $\hat y_i$:

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat y_i|$$
$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2}$$
$$R^2 = 1 - \frac{\sum_{i=1}^{n}(y_i-\hat y_i)^2}{\sum_{i=1}^{n}(y_i-\bar y)^2}$$

- **MAE:** sai lệch tuyệt đối trung bình, đơn vị đồng. Ví dụ 30.000 nghĩa là lệch trung bình 30.000đ/2 người.
- **RMSE:** đơn vị đồng; nhạy hơn với sai số lớn. MAE và RMSE càng thấp càng tốt.
- **R²:** tối đa 1, có thể âm; không phải phần trăm dự đoán đúng. R² = 0 tương ứng mốc trung bình
  của **chính tập đang đánh giá** trong công thức; âm là kém hơn mốc đó. Nếu mọi giá thật bằng nhau thì R² không xác định.
- **Mốc so sánh thực dụng:** luôn dự đoán bằng trung bình giá **train**, không nhìn giá test.
  Mốc này khác $\bar y$ trong định nghĩa R² trên test.
- In thêm kết quả train để nhận diện chênh lệch train/test, và riêng 42 quán Hòa Lạc trong test.
  Chênh lệch lớn giữa train/test có thể cho thấy overfitting.

Giữ nguyên dự đoán để chấm điểm, không cắt giá âm hoặc làm tròn trước khi tính chỉ số.

In [10]:
rows = []
test_predictions = {}
hoa_test = test_regions.eq("Cụm Hòa Lạc")

for name, model in models.items():
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    test_predictions[name] = test_pred
    for scope, truth, prediction in [
        ("Train", y_train, train_pred),
        ("Test toàn bộ", y_test, test_pred),
        ("Test cụm Hòa Lạc", y_test.loc[hoa_test], test_pred[hoa_test.to_numpy()])
    ]:
        rows.append({"Tập": scope, "Model": name, "Số quán": len(truth),
                     "MAE (VND)": mean_absolute_error(truth, prediction),
                     "RMSE (VND)": np.sqrt(mean_squared_error(truth, prediction)),
                     "R²": r2_score(truth, prediction, force_finite=False)})

for scope, truth in [("Test toàn bộ", y_test), ("Test cụm Hòa Lạc", y_test.loc[hoa_test])]:
    prediction = np.full(len(truth), y_train.mean())
    rows.append({"Tập": scope, "Model": "Trung bình train", "Số quán": len(truth),
                 "MAE (VND)": mean_absolute_error(truth, prediction),
                 "RMSE (VND)": np.sqrt(mean_squared_error(truth, prediction)),
                 "R²": r2_score(truth, prediction, force_finite=False)})

evaluation = pd.DataFrame(rows)
display(evaluation.round({"MAE (VND)": 0, "RMSE (VND)": 0, "R²": 4}))
test_results = data.loc[X_test.index, ["restaurant_id", "restaurant_name", "region", target]].copy()
test_results["pred_cost_lr"] = test_predictions["Linear Regression"]
test_results["pred_cost_rf"] = test_predictions["Random Forest"]
print("Ví dụ giá thực tế và dự đoán của các quán Hòa Lạc thuộc test:")
display(test_results.loc[hoa_test].head(10).round(0))
selected_test = evaluation[(evaluation["Tập"] == "Test toàn bộ") & (evaluation["Model"] == best_name)].iloc[0]
print(f"{best_name}: MAE test = {selected_test['MAE (VND)']:,.0f}đ; "
      f"RMSE = {selected_test['RMSE (VND)']:,.0f}đ; R² = {selected_test['R²']:.4f}.")

,Tập,Model,Số quán,MAE (VND),RMSE (VND),R²
0,Train,Linear Regression,968,37740.0,57648.0,0.9032
1,Test toàn bộ,Linear Regression,242,41298.0,64722.0,0.8758
2,Test cụm Hòa Lạc,Linear Regression,42,41458.0,62671.0,0.7216
3,Train,Random Forest,968,24308.0,40583.0,0.9520
4,Test toàn bộ,Random Forest,242,43060.0,72261.0,0.8451
5,Test cụm Hòa Lạc,Random Forest,42,45109.0,76747.0,0.5824
6,Test toàn bộ,Trung bình train,242,151454.0,183691.0,-0.0007
7,Test cụm Hòa Lạc,Trung bình train,42,137871.0,147657.0,-0.5457


Ví dụ giá thực tế và dự đoán của các quán Hòa Lạc thuộc test:


,restaurant_id,restaurant_name,region,average_cost_for_two,pred_cost_lr,pred_cost_rf
542,HN0543,Gà Rán & Tokbokki Bản Mường - Thạch Hòa 7,Cụm Hòa Lạc,155000,155406.0,161182.0
556,HN0557,Bún Bò Huế Xuân Hương - Thạch Hòa 21,Cụm Hòa Lạc,85000,132948.0,94904.0
1193,HN1194,Phở Gà Trộn 178 Km 29 Đại Lộ Thăng Long,Cụm Hòa Lạc,85000,91596.0,91137.0
538,HN0539,Nướng Chảo Gang Sơn Tây - Thạch Hòa 3,Cụm Hòa Lạc,195000,245973.0,128667.0
571,HN0572,Phở Bò Xuân Hương - Bắc Phú Cát 6,Cụm Hòa Lạc,95000,57645.0,102105.0
504,HN0505,Bún Riêu Cua Thái Sơn - Hòa Lạc 15,Cụm Hòa Lạc,65000,70060.0,106789.0
1185,HN1186,Tiệm Chè & Đồ Ăn Vặt Hương - Thôn 6,Cụm Hòa Lạc,70000,39865.0,71066.0
553,HN0554,Quán Nướng & Lẩu Sơn Tây - Thạch Hòa 18,Cụm Hòa Lạc,420000,363694.0,402644.0
451,HN0452,Quán Cháo Lòng Tiết Canh 371,Cụm Hòa Lạc,80000,177447.0,98199.0
487,HN0488,Phở Nhất Nhất,Cụm Hòa Lạc,65000,149745.0,95077.0


Random Forest: MAE test = 43,060đ; RMSE = 72,261đ; R² = 0.8451.


## Ô 11 — Tạo giá tham chiếu và danh sách quán giá thấp hơn dự báo
**Công dụng:** tạo kết quả cho toàn bộ 1.210 quán để đối chiếu, đặc biệt nhóm 210 quán cụm Hòa Lạc.

Để không xếp hạng bằng giá dự đoán ngay trên các quán đã dùng để fit, dùng **out-of-fold (OOF)** 5 lượt:
mỗi quán được dự đoán bởi một bản sao mô hình không học giá của chính quán đó trong lượt này.
Cấu hình hai mô hình và lựa chọn `best_name` giữ nguyên. OOF ở đây phục vụ lập bảng tham chiếu,
không thay thế chỉ số test ở Ô 10, không phải kết quả kiểm định chéo lồng nhau để đánh giá bước chọn mô hình.

$$resid\_pct = 100\times\frac{giá\ thực\ tế-pred\_cost}{pred\_cost}$$

`underpriced = True` khi giá thực thấp hơn dự báo **ít nhất 20%** và điểm đánh giá **≥ 4,2**.
Ví dụ giá thật 80.000đ, dự báo 100.000đ thì `resid_pct = -20%`.
Chỉ tính tỷ lệ khi giá dự đoán dương. Ngưỡng −20% là quy tắc của nhóm,
không phải xác suất hay một kết luận chắc chắn rằng quán đó giá “hời”.

In [11]:
oof_cv = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, data["region"]))
oof_predictions = {
    name: cross_val_predict(model, X, y, cv=oof_cv)
    for name, model in models.items()
}
results = df.set_index("restaurant_id").loc[data["restaurant_id"]].reset_index().copy()
results["aggregate_rating"] = data["aggregate_rating"].to_numpy()
results["region"] = data["region"].to_numpy()
results["pred_cost_lr"] = oof_predictions["Linear Regression"]
results["pred_cost_rf"] = oof_predictions["Random Forest"]
results["pred_cost"] = oof_predictions[best_name]
results["prediction_method"] = f"5-fold OOF / {best_name}"
results["resid_pct"] = np.nan
positive = results["pred_cost"] > 0
results.loc[positive, "resid_pct"] = (
    (results.loc[positive, target] - results.loc[positive, "pred_cost"])
    / results.loc[positive, "pred_cost"] * 100
)
results["underpriced"] = positive & (results["resid_pct"] <= -20) & (results["aggregate_rating"] >= 4.2)
hoa_results = results[results["region"] == "Cụm Hòa Lạc"].copy()
top_value = hoa_results[hoa_results["underpriced"]].sort_values("resid_pct").head(10)
print(f"Đã dự đoán OOF cho {len(results)} quán, trong đó {len(hoa_results)} quán cụm Hòa Lạc.")
print(f"Giá dự đoán không dương: {(~positive).sum()} quán; không gắn nhãn giá hời cho các quán này.")
print(f"Số quán cụm Hòa Lạc đạt điều kiện giá thấp hơn dự báo và rating ≥ 4,2: {hoa_results['underpriced'].sum()}")
display(top_value[["restaurant_id", "restaurant_name", "locality", target,
                   "pred_cost", "resid_pct", "aggregate_rating"]].round(2))

Đã dự đoán OOF cho 1210 quán, trong đó 210 quán cụm Hòa Lạc.
Giá dự đoán không dương: 0 quán; không gắn nhãn giá hời cho các quán này.
Số quán cụm Hòa Lạc đạt điều kiện giá thấp hơn dự báo và rating ≥ 4,2: 20


,restaurant_id,restaurant_name,locality,average_cost_for_two,pred_cost,resid_pct,aggregate_rating
1174,HN1175,Bún Đậu Mắm Tôm Ngọc - Thôn 6,Hòa Lạc,95000,302817.96,-68.63,4.2
552,HN0553,Lẩu Ếch & Lẩu Thái Ngọc Mai - Thạch Hòa 17,Thạch Hòa,185000,354050.54,-47.75,4.2
540,HN0541,Buffet Lẩu Nướng Tuấn Anh - Thạch Hòa 5,Thạch Hòa,320000,568540.71,-43.72,4.2
468,HN0469,Bánh Cuốn Thanh Đức,Hòa Lạc,60000,98447.74,-39.05,4.7
609,HN0610,Bún Thái Hải Sản Chua Cay,Tân Xã,85000,137619.25,-38.24,4.2
477,HN0478,Lẩu Sinh Viên 99k,Liên Quan,225000,358806.16,-37.29,4.7
579,HN0580,Nhà Hàng Hải Sản Đức Anh - Thạch Thất 4,Thạch Thất,330000,521276.15,-36.69,4.4
588,HN0589,Lẩu Ếch & Lẩu Thái Ngọc Mai - Liên Quan 5,Liên Quan,245000,370583.06,-33.89,4.3
484,HN0485,Hòa Lạc Quán,Thạch Hòa,280000,413544.42,-32.29,4.5
507,HN0508,Nhà Hàng Hải Sản Hương Quê - Hòa Lạc 18,Hòa Lạc,350000,505704.60,-30.79,4.2


## Ô 12 — Lưu các bảng kết quả
**Công dụng:** xuất kết quả để đưa vào report hoặc Power BI, không sửa bảng SQL hay file nguồn.

- `danh_gia_model.csv`: MAE, RMSE, R² và số mẫu của từng tập đánh giá.
- `du_doan_gia_oof.csv`: toàn bộ quán, giá dự đoán của hai mô hình, mô hình được chọn,
  sai số tương đối và nhãn `underpriced`; lọc `region = Cụm Hòa Lạc` để lấy 210 quán.
- `du_doan_test.csv`: giá dự đoán độc lập trên test của cả hai mô hình.

Các file nằm trong thư mục `ket_qua_regression` tại vị trí chạy notebook; chạy lại sẽ cập nhật các file kết quả này.
Giá trị được lưu đủ độ chính xác; chỉ làm tròn khi hiển thị.

In [12]:
output_dir = Path.cwd() / "ket_qua_regression"
output_dir.mkdir(exist_ok=True)
evaluation.to_csv(output_dir / "danh_gia_model.csv", index=False, encoding="utf-8-sig")
results.to_csv(output_dir / "du_doan_gia_oof.csv", index=False, encoding="utf-8-sig")
test_results.to_csv(output_dir / "du_doan_test.csv", index=False, encoding="utf-8-sig")
print("Đã lưu kết quả tại:", output_dir.resolve())

Đã lưu kết quả tại: C:\Users\NEO 16\Documents\Codex\2026-09-21\singular-value-decomposition-l-g\outputs\ket_qua_regression
